In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
from pathlib import Path
import cv2
from datetime import datetime
from honeybee_comb_inferer.inference import HoneyBeeCombInferer
from segmentation_restruct.comb_limitor import CombMaskGenerator
from segmentation_restruct.comb_limitor import CombMaskEvaluator

In [33]:
root_dir = Path().cwd().resolve().parent.parent
model_dir = root_dir  / 'models'
model_name = 'unet_effnetb0'
device = 'cuda'

model = model = HoneyBeeCombInferer(model_name=model_name, path_to_pretrained_models=model_dir, device=device)
generator = CombMaskGenerator(
    closing_kernel_size=11,
    closing_iterations=3,
    remove_outliers=True,      # NEW: Remove small isolated regions
    min_region_size=1000         # NEW: Minimum size (pixels) to keep a region
)
evaluator = CombMaskEvaluator()


In [34]:
date_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_name = f"matoschchuk_model_kernel11_iter3_with_suppression_min1000{date_str}"

print(f"Run name: {run_name}")


Run name: matoschchuk_model_kernel11_iter3_with_suppression_min10002025-10-25_18-31-38


In [35]:
images_dir = Path(r"E:\Bachelorarbeit\comb_limitation_dataset\images\test")
gt_dir = Path(r"E:\Bachelorarbeit\comb_limitation_dataset\labels\test")

image_files = sorted(images_dir.glob("*.png")) + sorted(images_dir.glob("*.jpg"))
print(f"Found {len(image_files)} images to process")

predicted_masks = []
ground_truth_masks = []
input_images = []
image_names = []

print("\nProcessing images...")
for img_file in image_files:
    # Find corresponding ground truth
    gt_file = gt_dir / img_file.name

    if not gt_file.exists():
        print(f"Skipping {img_file.name} - no ground truth found")
        continue

    print(f"Processing {img_file.name}...")

    input_img = cv2.imread(str(img_file), cv2.IMREAD_GRAYSCALE)

    seg_mask = model.infer(str(img_file))

    pred_mask = generator.generate_comb_mask(seg_mask)

    gt_mask = cv2.imread(str(gt_file), cv2.IMREAD_GRAYSCALE)

    predicted_masks.append(pred_mask)
    ground_truth_masks.append(gt_mask)
    input_images.append(input_img)
    image_names.append(img_file.name)

print(f"\nProcessed {len(predicted_masks)} images successfully")

Found 4 images to process

Processing images...
Processing 20240606_cam-3.png...
Processing background_cam-0_20250719T012105.601863.038Z.png...
Processing background_cam-3_20250705T043209.571836.965Z.png...
Processing k_background_cam-1_20250816T104412.249021.386Z--20250816T104512.171084.576Z.png...

Processed 4 images successfully


In [36]:
# Evaluate all masks at once
aggregate_metrics, per_image_metrics = evaluator.evaluate_batch_from_arrays(
    predicted_masks=predicted_masks,
    ground_truth_masks=ground_truth_masks,
    image_names=image_names
)

# Print aggregate results
evaluator.print_metrics(aggregate_metrics, title="Batch Evaluation Results")

# Print per-image results
print("\n" + "="*60)
print("Per-Image Results:")
print("="*60)
for metrics in per_image_metrics:
    print(f"\n{metrics['filename']}:")
    print(f"  IoU: {metrics['iou']:.4f}  |  Dice: {metrics['dice']:.4f}  |  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}  |  Recall: {metrics['recall']:.4f}")



                  Batch Evaluation Results                  

Aggregate Statistics (n=4 images):

  Main Metrics:
IoU:      0.9291 +/- 0.0448  (range: 0.8516 - 0.9573)
Dice/F1:  0.9627 +/- 0.0247  (range: 0.9199 - 0.9782)
Accuracy: 0.9553 +/- 0.0262  (range: 0.9099 - 0.9726)

  Precision & Recall:
Precision:   0.9927 +/- 0.0033  (range: 0.9884 - 0.9973)
Recall:      0.9354 +/- 0.0449  (range: 0.8580 - 0.9659)
Specificity: 0.9867 +/- 0.0074  (range: 0.9760 - 0.9965)

  Confusion Matrix Totals:
True Positives  (TP): 58,879,903  (avg: 14,719,976 per image)
True Negatives  (TN): 34,654,261  (avg: 8,663,565 per image)
False Positives (FP): 438,157  (avg: 109,539 per image)
False Negatives (FN): 3,938,463  (avg: 984,616 per image)
Total pixels:         97,910,784


Per-Image Results:

20240606_cam-3.png:
  IoU: 0.8516  |  Dice: 0.9199  |  Accuracy: 0.9099
  Precision: 0.9913  |  Recall: 0.8580

background_cam-0_20250719T012105.601863.038Z.png:
  IoU: 0.9551  |  Dice: 0.9770  |  Accuracy: 0.

In [37]:
saved_paths = evaluator.save_batch_results(
    aggregate_metrics=aggregate_metrics,
    per_image_metrics=per_image_metrics,
    output_dir=root_dir / "results",  # Results will be saved in project/results/
    run_name=run_name,  # Using the run_name defined above
    save_figure=True,
    predicted_masks=predicted_masks,
    ground_truth_masks=ground_truth_masks,
    input_images=input_images,
    image_names=image_names
)

print("\nSaved files:")
for key, path in saved_paths.items():
    print(f"  {key}: {path.name}")

✓ Aggregate metrics saved to: C:\Users\sturmd\Documents\Privates\honeybee_cells_segmentation_inference\results\matoschchuk_model_kernel11_iter3_with_suppression_min10002025-10-25_18-31-38\aggregate_metrics.json
✓ Per-image metrics saved to: C:\Users\sturmd\Documents\Privates\honeybee_cells_segmentation_inference\results\matoschchuk_model_kernel11_iter3_with_suppression_min10002025-10-25_18-31-38\per_image_metrics.json
✓ Visualization saved to: C:\Users\sturmd\Documents\Privates\honeybee_cells_segmentation_inference\results\matoschchuk_model_kernel11_iter3_with_suppression_min10002025-10-25_18-31-38\visualization.png
✓ Summary saved to: C:\Users\sturmd\Documents\Privates\honeybee_cells_segmentation_inference\results\matoschchuk_model_kernel11_iter3_with_suppression_min10002025-10-25_18-31-38\summary.txt

All results saved to: C:\Users\sturmd\Documents\Privates\honeybee_cells_segmentation_inference\results\matoschchuk_model_kernel11_iter3_with_suppression_min10002025-10-25_18-31-38


Sav